# Safety Kit Detection: AI-Powered PPE Compliance Monitoring for Construction Sites

This dataset addresses the critical challenge of automated safety compliance monitoring in construction environments through a sophisticated personal protective equipment (PPE) detection system. Our project focuses on five essential safety items: **helmets**, **safety vests**, **boots**, **gloves**, and **goggles**.

---

## Dataset Overview

We've curated a specialized collection of construction site images through meticulous manual selection. While numerous construction safety datasets exist, finding images of workers wearing all required safety equipment simultaneously proved challenging. We addressed this gap by carefully selecting and sorting images that represent both compliant and non-compliant scenarios, making this dataset particularly valuable for real-world applications.

---

## Unique Features

- Multi-class PPE detection covering complete safety gear
- Carefully curated images from diverse construction environments
- Balanced mix of compliant and non-compliant scenarios
- Real-world construction site conditions
- Various lighting conditions and worker positions

---

## Labeling System

Our dataset employs a comprehensive **dual-positive and negative labeling** approach with **11 distinct classes**:

### Positive Classes (Equipment Present)

| Class ID | Label   |
|----------|---------|
| 0        | Helmet  |
| 1        | Gloves  |
| 2        | Vest    |
| 3        | Boots   |
| 4        | Goggles |
| 6        | Person  |

### Negative Classes (Missing Equipment)

| Class ID | Label      |
|----------|------------|
| 7        | no_helmet  |
| 8        | no_goggle  |
| 9        | no_gloves  |
| 10       | no_boots   |

### Additional

| Class ID | Label | Description                        |
|----------|-------|------------------------------------|
| 5        | none  | For areas with no relevant objects |

---

This structured labeling system enables models to not only detect properly worn safety equipment but also explicitly identify missing items, making it ideal for practical safety monitoring applications. The dataset supports the development of automated systems that can enhance workplace safety by providing **real-time monitoring of PPE compliance** in construction environments.

---

**Dataset Source:** [Kaggle - PPE Kit Detection (Construction Site Workers)](https://www.kaggle.com/datasets/ketakichalke/ppe-kit-detection-construction-site-workers)

## 📦 Installation & Imports

In [ ]:
# Install required packages
!pip install ultralytics imagehash albumentations gradio --quiet

# ── System / File handling ──────────────────────────────────────────────────
import os, glob, random, shutil, json, time, logging, warnings
from collections import Counter, defaultdict
from shutil import copyfile

warnings.filterwarnings("ignore")

# ── Data manipulation & visualization ───────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from scipy.spatial.distance import jensenshannon

# ── Image processing ─────────────────────────────────────────────────────────
import cv2
from PIL import Image
import imagehash
import albumentations as A

# ── YOLO ─────────────────────────────────────────────────────────────────────
import ultralytics
from ultralytics import YOLO
import yaml

ultralytics.checks()
print("✅ All libraries loaded successfully!")

## ⚙️ Configuration

Central configuration class — set `DEBUG = True` for quick sanity-check runs.

In [ ]:
class CFG:
    # ── Debug / Reproducibility ────────────────────────────────────────────
    DEBUG   = False          # set True for fast trial runs
    SEED    = 42

    # ── Dataset classes (Ketaki PPE dataset — 11 classes from original labels)
    # Class 5 ("none") is kept so bounding-box IDs stay aligned with label files.
    PPE_CLASSES = {
        0:  "Helmet",
        1:  "Gloves",
        2:  "Vest",
        3:  "Boots",
        4:  "Goggles",
        5:  "none",
        6:  "Person",
        7:  "no_helmet",
        8:  "no_goggle",
        9:  "no_gloves",
        10: "no_boots",
    }
    NUM_CLASSES = len(PPE_CLASSES)

    # Semantic groupings used for colour-coded visualisations
    POSITIVE_CLASSES = {"Helmet", "Gloves", "Vest", "Boots", "Goggles", "Person"}
    NEGATIVE_CLASSES = {"no_helmet", "no_goggle", "no_gloves", "no_boots"}

    # ── Split ratios ───────────────────────────────────────────────────────
    TARGET_RATIOS = {"train": 0.75, "valid": 0.15, "test": 0.10}

    # ── Training hyper-parameters ──────────────────────────────────────────
    EPOCHS      = 3  if DEBUG else 50
    PATIENCE    = 10 if DEBUG else 20
    BATCH_SIZE  = 4  if DEBUG else 16
    IMGSZ       = 640
    FRACTION    = 0.05 if DEBUG else 1.0
    BASE_MODEL  = "yolo11n.pt"   # switch to yolo11s.pt / yolo11m.pt for better accuracy
    DEVICE      = "0"            # GPU index; use "cpu" if no GPU
    LR0         = 0.01
    COS_LR      = True
    OPTIMIZER   = "AdamW"
    RECT        = False

    # ── Paths ─────────────────────────────────────────────────────────────
    DATA_PATH    = "/kaggle/input/ppe-kit-detection-construction-site-workers/data"
    WORKING_PATH = "/kaggle/working"
    FOLDERS      = ["train", "valid", "test"]

    # Derived output paths
    YAML_PATH          = f"{WORKING_PATH}/dataset.yaml"
    TRAIN_RESULTS_CSV  = f"{WORKING_PATH}/train_results.csv"
    MODEL_PATH         = f"{WORKING_PATH}/runs/detect/ppe_run/weights/best.pt"
    OUTPUT_MODEL_PATH  = f"{WORKING_PATH}/best_ppe_model.pt"
    METADATA_PATH      = f"{WORKING_PATH}/model_metadata.json"
    LOG_PATH           = f"{WORKING_PATH}/training_log.txt"

    @staticmethod
    def set_seed():
        random.seed(CFG.SEED)
        np.random.seed(CFG.SEED)
        os.environ["PYTHONHASHSEED"] = str(CFG.SEED)

CFG.set_seed()
print(f"📌 Classes  : {CFG.NUM_CLASSES}")
print(f"📌 Debug    : {CFG.DEBUG}")
print(f"📌 Model    : {CFG.BASE_MODEL}")
print(f"📌 Epochs   : {CFG.EPOCHS}")
print(f"📌 Data path: {CFG.DATA_PATH}")


## 🗂️ Setup — Working Directory & Logger

In [ ]:
# ── Logger ────────────────────────────────────────────────────────────────────
def setup_logger():
    logger = logging.getLogger("ppe_detection")
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        fh = logging.FileHandler(CFG.LOG_PATH)
        fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s",
                                          datefmt="%Y-%m-%d %H:%M:%S"))
        logger.addHandler(fh)
    return logger

logger = setup_logger()

# ── Clean /kaggle/working/ ────────────────────────────────────────────────────
def clean_working_directory():
    logger.info("Cleaning working directory")
    for item in os.listdir(CFG.WORKING_PATH):
        path = os.path.join(CFG.WORKING_PATH, item)
        try:
            if os.path.isfile(path) or os.path.islink(path):
                os.unlink(path)
            elif os.path.isdir(path):
                shutil.rmtree(path)
        except Exception as e:
            logger.error(f"Failed to delete {path}: {e}")
    print("✅ /kaggle/working/ cleaned")

# ── Create sub-folder structure ───────────────────────────────────────────────
def create_folder_structure():
    for folder in CFG.FOLDERS:
        os.makedirs(os.path.join(CFG.WORKING_PATH, folder, "images"), exist_ok=True)
        os.makedirs(os.path.join(CFG.WORKING_PATH, folder, "labels"), exist_ok=True)
    print("✅ Folder structure created")
    os.makedirs(os.path.join(CFG.WORKING_PATH, "runs"), exist_ok=True)

clean_working_directory()
create_folder_structure()


---
## 🔍 Exploratory Data Analysis (EDA)

### 1 — Dataset Inventory

In [ ]:
# ── Discover raw dataset folders ──────────────────────────────────────────────
# The Ketaki dataset may use "train / val / test" or "train / valid / test"
RAW_FOLDERS = []
for candidate in ["train", "val", "valid", "test"]:
    img_dir = os.path.join(CFG.DATA_PATH, "images", candidate)
    if os.path.exists(img_dir):
        RAW_FOLDERS.append(candidate)

print(f"📂 Found raw split folders: {RAW_FOLDERS}\n")

# ── Count images & labels per split ──────────────────────────────────────────
summary_rows = []
for folder in RAW_FOLDERS:
    img_dir   = os.path.join(CFG.DATA_PATH, "images", folder)
    lbl_dir   = os.path.join(CFG.DATA_PATH, "labels", folder)
    images    = set(os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".png", ".jpeg")))
    labels    = set(os.path.splitext(f)[0] for f in os.listdir(lbl_dir) if f.endswith(".txt"))
    missing_l = images - labels
    missing_i = labels - images
    summary_rows.append({
        "Split": folder, "Images": len(images), "Labels": len(labels),
        "Missing Labels": len(missing_l), "Missing Images": len(missing_i)
    })
    print(f"📂 {folder.upper():6s}  images={len(images):5d}  labels={len(labels):5d}  "
          f"missing_labels={len(missing_l)}  missing_images={len(missing_i)}")

df_inventory = pd.DataFrame(summary_rows)
print("\n", df_inventory.to_string(index=False))


### 2 — Build Master DataFrame & Class Distribution

In [ ]:
def build_master_df(data_path, raw_folders, binary=True):
    """
    Build a DataFrame where each row = one image.
    Columns: filename | origin | 0..10 (class presence/count).
    binary=True → 1 if class present, 0 otherwise (recommended for stratification).
    """
    records = []
    for folder in raw_folders:
        lbl_dir = os.path.join(data_path, "labels", folder)
        for lbl_file in glob.glob(os.path.join(lbl_dir, "*.txt")):
            img_name = os.path.splitext(os.path.basename(lbl_file))[0]
            class_vals = {cls: 0 for cls in CFG.PPE_CLASSES}
            with open(lbl_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    cid = int(parts[0])
                    if cid in CFG.PPE_CLASSES:
                        class_vals[cid] = 1 if binary else class_vals[cid] + 1
            row = {"filename": img_name, "origin": folder}
            row.update(class_vals)
            records.append(row)
    return pd.DataFrame(records)

df_total = build_master_df(CFG.DATA_PATH, RAW_FOLDERS, binary=True)
print(f"📊 Total images: {len(df_total)}")
print(df_total.head())


In [ ]:
# ── Raw class frequency across all splits ─────────────────────────────────────
df_count = build_master_df(CFG.DATA_PATH, RAW_FOLDERS, binary=False)
class_cols = list(CFG.PPE_CLASSES.keys())

total_counts = df_count[class_cols].sum()
class_names  = [CFG.PPE_CLASSES[c] for c in class_cols]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar — raw count
colors = ["#2ECC71" if CFG.PPE_CLASSES[c] in CFG.POSITIVE_CLASSES
          else "#E74C3C" if CFG.PPE_CLASSES[c] in CFG.NEGATIVE_CLASSES
          else "#95A5A6" for c in class_cols]
axes[0].bar(class_names, total_counts.values, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_title("Total Annotation Count per Class", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Class"); axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

# Pie — proportion
wedge_colors = colors
axes[1].pie(total_counts.values, labels=class_names, colors=wedge_colors,
            autopct="%1.1f%%", startangle=140,
            textprops={"fontsize": 8})
axes[1].set_title("Class Distribution (Proportion)", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(CFG.WORKING_PATH, "class_distribution.png"), dpi=120, bbox_inches="tight")
plt.show()
print("🟢 Positive classes | 🔴 Negative (missing PPE) | ⚪ Neutral")


### 3 — Sample Images with Ground-Truth Bounding Boxes

In [ ]:
COLOR_MAP = {
    "Helmet": (0, 200, 0),    "Gloves": (0, 150, 255),
    "Vest":   (255, 200, 0),  "Boots":  (0, 255, 255),
    "Goggles":(200, 0, 255),  "Person": (100, 100, 255),
    "none":   (180, 180, 180),
    "no_helmet": (255, 0, 0), "no_goggle": (200, 0, 100),
    "no_gloves": (255, 80, 0),"no_boots": (150, 0, 0),
}

def draw_yolo_boxes(image_path, label_path, class_map=CFG.PPE_CLASSES):
    img   = cv2.imread(image_path)
    if img is None:
        return None
    img   = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w  = img.shape[:2]
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cid = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                x1 = int((xc - bw / 2) * w); y1 = int((yc - bh / 2) * h)
                x2 = int((xc + bw / 2) * w); y2 = int((yc + bh / 2) * h)
                name  = class_map.get(cid, str(cid))
                color = COLOR_MAP.get(name, (128, 128, 128))
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cv2.putText(img, name, (x1, max(y1 - 5, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return img

# Show 6 random images from first raw folder
sample_folder = RAW_FOLDERS[0]
img_dir = os.path.join(CFG.DATA_PATH, "images", sample_folder)
lbl_dir = os.path.join(CFG.DATA_PATH, "labels", sample_folder)

all_imgs = [f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
samples  = random.sample(all_imgs, min(6, len(all_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for ax, fname in zip(axes.flatten(), samples):
    base    = os.path.splitext(fname)[0]
    img_rgb = draw_yolo_boxes(os.path.join(img_dir, fname),
                              os.path.join(lbl_dir, base + ".txt"))
    if img_rgb is not None:
        ax.imshow(img_rgb)
    ax.axis("off")
    ax.set_title(fname[:40], fontsize=7)

plt.suptitle(f"Sample Images from '{sample_folder}' split with GT Boxes", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CFG.WORKING_PATH, "sample_images.png"), dpi=120, bbox_inches="tight")
plt.show()


---
## 🔀 Data Preparation

### 1 — Stratified Re-split

We pool **all** images and re-split them at `75 / 15 / 10` (train/valid/test) using **binary class presence** as the stratification key.  
This fixes the original imbalanced distribution shown above.

In [ ]:
def stratified_split(df, target_ratios, seed=CFG.SEED):
    """Simple sequential stratified split keyed on binary class presence."""
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    n  = len(df)
    n_train = int(n * target_ratios["train"])
    n_valid = int(n * target_ratios["valid"])
    df_train = df.iloc[:n_train].copy()
    df_valid = df.iloc[n_train : n_train + n_valid].copy()
    df_test  = df.iloc[n_train + n_valid:].copy()
    return df_train, df_valid, df_test

df_train, df_valid, df_test = stratified_split(df_total, CFG.TARGET_RATIOS)

total = len(df_train) + len(df_valid) + len(df_test)
print(f"✅ Total  : {total} images")
print(f"   Train  : {len(df_train)} ({len(df_train)/total:.1%})")
print(f"   Valid  : {len(df_valid)} ({len(df_valid)/total:.1%})")
print(f"   Test   : {len(df_test)}  ({len(df_test)/total:.1%})")

# Verify no data loss
assert total == len(df_total), "⚠️ Data loss during split!"
print("✅ No images lost in split.")


In [ ]:
# ── Plot distribution BEFORE vs AFTER re-split ────────────────────────────────
def class_pct(df):
    n = len(df)
    return {CFG.PPE_CLASSES[c]: df[c].sum() / n * 100 for c in CFG.PPE_CLASSES}

splits = {"Train (new)": class_pct(df_train),
          "Valid (new)": class_pct(df_valid),
          "Test  (new)": class_pct(df_test)}

df_dist = pd.DataFrame([
    {"Class": cls, "Percentage": pct, "Split": split}
    for split, data in splits.items()
    for cls, pct in data.items()
])

plt.figure(figsize=(14, 6))
sns.barplot(data=df_dist, x="Class", y="Percentage", hue="Split", palette="Set2")
plt.title("Class Distribution After Stratified Re-split", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right"); plt.tight_layout()
plt.savefig(os.path.join(CFG.WORKING_PATH, "distribution_after_split.png"), dpi=120, bbox_inches="tight")
plt.show()

# Jensen-Shannon divergence: how similar are the splits?
def arr(d): return np.array(list(d.values()), dtype=float) + 1e-9
t, v, te = arr(class_pct(df_train)), arr(class_pct(df_valid)), arr(class_pct(df_test))
print(f"\n📊 Jensen-Shannon Divergence (lower = more similar distributions):")
print(f"   Train vs Valid : {jensenshannon(t, v):.4f}")
print(f"   Train vs Test  : {jensenshannon(t, te):.4f}")


### 2 — Copy Files into Working Directory

In [ ]:
EXTS = [".jpg", ".jpeg", ".png", ".bmp"]

def move_files(df: pd.DataFrame, subset: str, raw_data_path: str = CFG.DATA_PATH) -> dict:
    """Copy images + labels from the raw Kaggle input to the working directory."""
    dst_img  = os.path.join(CFG.WORKING_PATH, subset, "images")
    dst_lbl  = os.path.join(CFG.WORKING_PATH, subset, "labels")
    counters = {"copied": 0, "missing_img": 0, "missing_lbl": 0}

    for _, row in df.iterrows():
        stem   = Path(row["filename"]).stem
        origin = row["origin"]          # e.g. "train" / "val" / "test"

        # ── locate image ──────────────────────────────────────────────────────
        img_src = None
        for ext in EXTS:
            candidate = os.path.join(raw_data_path, "images", origin, stem + ext)
            if os.path.exists(candidate):
                img_src = candidate
                break
        if img_src is None:
            counters["missing_img"] += 1
            continue

        # ── locate label ──────────────────────────────────────────────────────
        lbl_src = os.path.join(raw_data_path, "labels", origin, stem + ".txt")
        if not os.path.exists(lbl_src):
            counters["missing_lbl"] += 1
            continue

        shutil.copy(img_src, os.path.join(dst_img, os.path.basename(img_src)))
        shutil.copy(lbl_src, os.path.join(dst_lbl, stem + ".txt"))
        counters["copied"] += 1

    return counters


for subset, df_sub in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    stats = move_files(df_sub, subset)
    print(f"[{subset:>5}]  copied={stats['copied']:4d}  "
          f"missing_img={stats['missing_img']}  missing_lbl={stats['missing_lbl']}")

print("\n✅ All files copied.")


### 3 — Duplicate Detection & Removal (pHash)

In [ ]:
def find_and_remove_duplicates(base_path: str, hash_thresh: int = 8) -> int:
    """
    Compare every image in train/valid/test with pHash.
    Removes duplicates (keeps the first occurrence) and the matching label file.
    Returns the number of removed pairs.
    """
    hash_map: dict = {}   # hash_str → first image path
    removed = 0

    for subset in CFG.FOLDERS:
        img_dir = os.path.join(base_path, subset, "images")
        lbl_dir = os.path.join(base_path, subset, "labels")
        if not os.path.isdir(img_dir):
            continue

        for img_path in sorted(glob.glob(os.path.join(img_dir, "*"))):
            try:
                h = str(imagehash.phash(Image.open(img_path)))
            except Exception:
                continue

            if h in hash_map:
                os.remove(img_path)
                lbl_path = os.path.join(lbl_dir, Path(img_path).stem + ".txt")
                if os.path.exists(lbl_path):
                    os.remove(lbl_path)
                removed += 1
            else:
                hash_map[h] = img_path

    return removed

n_removed = find_and_remove_duplicates(CFG.WORKING_PATH)
print(f"🗑️  Duplicate images removed: {n_removed}")

# Re-count files after deduplication
for subset in CFG.FOLDERS:
    imgs = glob.glob(os.path.join(CFG.WORKING_PATH, subset, "images", "*"))
    lbls = glob.glob(os.path.join(CFG.WORKING_PATH, subset, "labels", "*.txt"))
    print(f"   [{subset:>5}]  images={len(imgs)}  labels={len(lbls)}")


## 🔧 Data Augmentation

Underrepresented negative-PPE classes (`no_goggle`, `no_boots`, `no_helmet`) are synthetically
expanded in the **training split only** using albumentations:  
horizontal flip • random brightness/contrast • hue-saturation shift • Gaussian noise.  
YOLO labels are kept fully compatible (bounding boxes re-computed after flipping).

In [ ]:
import albumentations as A

AUG_PIPELINE = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=0.5),
        A.GaussNoise(var_limit=(10, 50), p=0.3),
        A.CLAHE(clip_limit=2.0, p=0.3),
    ],
    bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.3),
)


def read_yolo_labels(lbl_path: str):
    if not os.path.exists(lbl_path):
        return [], []
    rows = []
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                rows.append([int(parts[0])] + [float(x) for x in parts[1:]])
    if not rows:
        return [], []
    arr = np.array(rows)
    return arr[:, 0].astype(int).tolist(), arr[:, 1:].tolist()


def augment_image(img_path: str, lbl_path: str, out_img_dir: str, out_lbl_dir: str,
                  n_copies: int = 2) -> int:
    image = cv2.imread(img_path)
    if image is None:
        return 0
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    classes, bboxes = read_yolo_labels(lbl_path)
    stem   = Path(img_path).stem
    ext    = Path(img_path).suffix
    saved  = 0

    for i in range(n_copies):
        try:
            aug = AUG_PIPELINE(image=image, bboxes=bboxes, class_labels=classes)
        except Exception:
            continue

        out_stem = f"{stem}_aug{i}"
        out_img  = os.path.join(out_img_dir, out_stem + ext)
        out_lbl  = os.path.join(out_lbl_dir, out_stem + ".txt")

        cv2.imwrite(out_img, cv2.cvtColor(aug["image"], cv2.COLOR_RGB2BGR))
        with open(out_lbl, "w") as f:
            for cls, bb in zip(aug["class_labels"], aug["bboxes"]):
                f.write(f"{cls} {bb[0]:.6f} {bb[1]:.6f} {bb[2]:.6f} {bb[3]:.6f}\n")
        saved += 1
    return saved


# ── Identify which classes need augmentation ──────────────────────────────────
train_img_dir = os.path.join(CFG.WORKING_PATH, "train", "images")
train_lbl_dir = os.path.join(CFG.WORKING_PATH, "train", "labels")

# Count annotations per class in training set
from collections import Counter
class_counter: Counter = Counter()
for lbl_file in glob.glob(os.path.join(train_lbl_dir, "*.txt")):
    cls_ids, _ = read_yolo_labels(lbl_file)
    class_counter.update(cls_ids)

print("Training annotation counts per class:")
for cid, name in CFG.PPE_CLASSES.items():
    print(f"   {cid:2d}  {name:<15} {class_counter.get(cid, 0):>5}")

median_count = np.median(list(class_counter.values()))
TARGET_CLASSES_AUG = {
    cid for cid, cnt in class_counter.items()
    if cnt < median_count * 0.5
}
print(f"\n📌 Classes below 50 % of median ({median_count:.0f}) → will be augmented:")
for cid in sorted(TARGET_CLASSES_AUG):
    print(f"   {cid}  {CFG.PPE_CLASSES[cid]}")

# ── Run augmentation ──────────────────────────────────────────────────────────
total_aug = 0
if TARGET_CLASSES_AUG:
    for lbl_file in glob.glob(os.path.join(train_lbl_dir, "*.txt")):
        cls_ids, _ = read_yolo_labels(lbl_file)
        if any(c in TARGET_CLASSES_AUG for c in cls_ids):
            stem    = Path(lbl_file).stem
            img_src = None
            for ext in EXTS:
                cand = os.path.join(train_img_dir, stem + ext)
                if os.path.exists(cand):
                    img_src = cand
                    break
            if img_src:
                total_aug += augment_image(img_src, lbl_file,
                                           train_img_dir, train_lbl_dir, n_copies=2)

print(f"\n✅ Augmented images added: {total_aug}")
imgs_after = glob.glob(os.path.join(train_img_dir, "*"))
print(f"   Training images (after augmentation): {len(imgs_after)}")


## 🧠 Model Training

### YOLO11 — Architecture Overview

| Variant | Params | mAP50 (COCO) | Use-case |
|---------|--------|--------------|----------|
| yolo11n | ~2.6 M | 39.5 | Edge / fast inference |
| yolo11s | ~9.4 M | 47.0 | Balanced speed/accuracy |
| yolo11m | ~20.1 M | 51.5 | High accuracy |

We use **YOLO11n** as the base model — sufficient for 11-class PPE detection at 640 px,
and fast to train within Kaggle's GPU quota.

### 1 — Create Dataset YAML

In [ ]:
dataset_config = {
    "path":  CFG.WORKING_PATH,
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    CFG.NUM_CLASSES,
    "names": [CFG.PPE_CLASSES[i] for i in range(CFG.NUM_CLASSES)],
}

with open(CFG.YAML_PATH, "w") as f:
    yaml.dump(dataset_config, f, default_flow_style=False, sort_keys=False)

print(f"✅ YAML saved → {CFG.YAML_PATH}")
print(yaml.dump(dataset_config, default_flow_style=False, sort_keys=False))


### 2 — Train

In [ ]:
from ultralytics import YOLO

CFG.set_seed()
model = YOLO(CFG.BASE_MODEL)

print(f"🚀 Training  {CFG.BASE_MODEL}  for {CFG.EPOCHS} epochs  (imgsz={CFG.IMGSZ})")
print(f"   Data yaml : {CFG.YAML_PATH}")
print(f"   Device    : {CFG.DEVICE}")
print(f"   Batch     : {CFG.BATCH_SIZE}")

train_results = model.train(
    data      = CFG.YAML_PATH,
    epochs    = CFG.EPOCHS,
    patience  = CFG.PATIENCE,
    batch     = CFG.BATCH_SIZE,
    imgsz     = CFG.IMGSZ,
    device    = CFG.DEVICE,
    optimizer = CFG.OPTIMIZER,
    lr0       = CFG.LR0,
    cos_lr    = CFG.COS_LR,
    seed      = CFG.SEED,
    fraction  = CFG.FRACTION,
    rect      = CFG.RECT,
    project   = CFG.WORKING_PATH,
    name      = "ppe_yolo11n",
    exist_ok  = True,
    verbose   = True,
)

print("\n✅ Training complete.")


## 📊 Evaluation

### 1 — Training Curves

In [ ]:
run_dir = os.path.join(CFG.WORKING_PATH, "ppe_yolo11n")

# ── Load results CSV ──────────────────────────────────────────────────────────
results_csv = os.path.join(run_dir, "results.csv")
if os.path.exists(results_csv):
    df_res = pd.read_csv(results_csv)
    df_res.columns = df_res.columns.str.strip()

    metrics = {
        "Box Loss (train)":  ("train/box_loss",  "#e74c3c"),
        "Box Loss (val)":    ("val/box_loss",     "#c0392b"),
        "Cls Loss (train)":  ("train/cls_loss",   "#3498db"),
        "Cls Loss (val)":    ("val/cls_loss",      "#2980b9"),
        "mAP50":             ("metrics/mAP50(B)",  "#2ecc71"),
        "mAP50-95":          ("metrics/mAP50-95(B)", "#27ae60"),
    }

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("YOLO11n Training Curves", fontsize=16, fontweight="bold")

    for ax, (group, pairs) in zip(axes, [
        [("Box Loss (train)", "Box Loss (val)")],
        [("Cls Loss (train)", "Cls Loss (val)")],
        [("mAP50", "mAP50-95")],
    ]):
        for label in pairs[0]:
            col, color = metrics[label]
            if col in df_res.columns:
                ax.plot(df_res["epoch"], df_res[col], label=label, color=color, linewidth=2)
        ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)

    axes[0].set_title("Box Loss"); axes[1].set_title("Class Loss"); axes[2].set_title("mAP")
    plt.tight_layout()
    out_path = os.path.join(CFG.WORKING_PATH, "training_curves.png")
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"📈 Saved → {out_path}")
else:
    print(f"⚠️  results.csv not found at {results_csv}")


### 2 — Validation & Test Metrics

In [ ]:
best_weights = os.path.join(run_dir, "weights", "best.pt")
trained_model = YOLO(best_weights)

# ── Validation set ────────────────────────────────────────────────────────────
val_metrics = trained_model.val(data=CFG.YAML_PATH, split="val",
                                 imgsz=CFG.IMGSZ, device=CFG.DEVICE, verbose=False)

# ── Test set ──────────────────────────────────────────────────────────────────
test_metrics = trained_model.val(data=CFG.YAML_PATH, split="test",
                                  imgsz=CFG.IMGSZ, device=CFG.DEVICE, verbose=False)

def print_metrics(tag, m):
    print(f"\n{'─'*50}")
    print(f"  {tag}")
    print(f"{'─'*50}")
    print(f"  Precision : {m.box.mp:.4f}")
    print(f"  Recall    : {m.box.mr:.4f}")
    print(f"  mAP50     : {m.box.map50:.4f}")
    print(f"  mAP50-95  : {m.box.map:.4f}")
    print()
    rows = []
    for i, name in enumerate(CFG.PPE_CLASSES.values()):
        try:
            rows.append({
                "Class": name,
                "P":     round(float(m.box.p[i]), 3),
                "R":     round(float(m.box.r[i]), 3),
                "mAP50": round(float(m.box.ap50[i]), 3),
                "mAP50-95": round(float(m.box.ap[i]), 3),
            })
        except IndexError:
            pass
    df_m = pd.DataFrame(rows)
    print(df_m.to_string(index=False))

print_metrics("📋 VALIDATION SET", val_metrics)
print_metrics("📋 TEST SET",        test_metrics)

# ── Save summary CSV ──────────────────────────────────────────────────────────
summary = {
    "val_mAP50":     val_metrics.box.map50,
    "val_mAP50-95":  val_metrics.box.map,
    "test_mAP50":    test_metrics.box.map50,
    "test_mAP50-95": test_metrics.box.map,
}
pd.DataFrame([summary]).to_csv(CFG.TRAIN_RESULTS_CSV, index=False)
print(f"\n✅ Metrics saved → {CFG.TRAIN_RESULTS_CSV}")


### 3 — Confusion Matrix & YOLO Result Artefacts

In [ ]:
artefact_images = {
    "Confusion Matrix (norm)": os.path.join(run_dir, "confusion_matrix_normalized.png"),
    "Confusion Matrix":        os.path.join(run_dir, "confusion_matrix.png"),
    "F1 Curve":                os.path.join(run_dir, "F1_curve.png"),
    "PR Curve":                os.path.join(run_dir, "PR_curve.png"),
    "P Curve":                 os.path.join(run_dir, "P_curve.png"),
    "R Curve":                 os.path.join(run_dir, "R_curve.png"),
    "Labels":                  os.path.join(run_dir, "labels.jpg"),
}

found = {k: v for k, v in artefact_images.items() if os.path.exists(v)}
if not found:
    print("⚠️  No artefact images found yet (run training first).")
else:
    cols = 3
    rows = (len(found) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 7, rows * 6))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for ax, (title, path) in zip(axes, found.items()):
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        ax.imshow(img); ax.set_title(title, fontsize=11, fontweight="bold")
        ax.axis("off")
    for ax in axes[len(found):]:
        ax.axis("off")

    plt.tight_layout()
    out = os.path.join(CFG.WORKING_PATH, "training_artefacts.png")
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"📊 Saved → {out}")


### 4 — Inference Visualisation on Test Set

Boxes are colour-coded:  
🟢 **Green** → PPE worn (Helmet, Gloves, Vest, Boots, Goggles, Person)  
🔴 **Red** → PPE missing (no_helmet, no_goggle, no_gloves, no_boots)  
⬜ **Grey** → *none* class (background / unclassified)

In [ ]:
BOX_COLORS = {
    "positive": (0,   200,  80),   # green
    "negative": (220,  40,  40),   # red
    "neutral":  (160, 160, 160),   # grey
}

def box_color(class_name: str) -> tuple:
    if class_name in CFG.POSITIVE_CLASSES:
        return BOX_COLORS["positive"]
    if class_name in CFG.NEGATIVE_CLASSES:
        return BOX_COLORS["negative"]
    return BOX_COLORS["neutral"]


def draw_predictions(image: np.ndarray, result) -> tuple[np.ndarray, dict]:
    """Draw predicted boxes on image; return annotated image + compliance dict."""
    img = image.copy()
    h, w = img.shape[:2]
    compliance = {"wearing": [], "missing": []}

    if result.boxes is None or len(result.boxes) == 0:
        return img, compliance

    for box in result.boxes:
        cls_id    = int(box.cls[0])
        conf      = float(box.conf[0])
        cls_name  = CFG.PPE_CLASSES.get(cls_id, "unknown")
        color     = box_color(cls_name)
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = f"{cls_name} {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(img, (x1, y1 - th - 6), (x1 + tw, y1), color, -1)
        cv2.putText(img, label, (x1, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)

        if cls_name in CFG.NEGATIVE_CLASSES:
            compliance["missing"].append(cls_name)
        elif cls_name in CFG.POSITIVE_CLASSES:
            compliance["wearing"].append(cls_name)
    return img, compliance


# ── Pick random test images ───────────────────────────────────────────────────
test_img_dir = os.path.join(CFG.WORKING_PATH, "test", "images")
test_imgs    = glob.glob(os.path.join(test_img_dir, "*"))
sample_imgs  = random.sample(test_imgs, min(8, len(test_imgs)))

n_cols = 4
n_rows = (len(sample_imgs) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 5))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_imgs):
    image   = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    results = trained_model(img_path, verbose=False)
    annotated, comp = draw_predictions(image, results[0])

    status = "✅ Compliant" if not comp["missing"] else f"❌ Missing: {', '.join(set(comp['missing']))}"
    ax.imshow(annotated)
    ax.set_title(f"{Path(img_path).name[:22]}\n{status}", fontsize=8)
    ax.axis("off")

for ax in axes[len(sample_imgs):]:
    ax.axis("off")

plt.suptitle("Test-Set Predictions (green=PPE worn, red=PPE missing)", fontsize=13, fontweight="bold")
plt.tight_layout()
out = os.path.join(CFG.WORKING_PATH, "test_predictions.png")
plt.savefig(out, dpi=120, bbox_inches="tight")
plt.show()
print(f"🖼️  Saved → {out}")


## 💾 Save Model & Metadata

In [ ]:
import shutil

# Copy best.pt to the designated output path
shutil.copy(best_weights, CFG.OUTPUT_MODEL_PATH)
print(f"✅ Model saved → {CFG.OUTPUT_MODEL_PATH}")

# ── Metadata JSON ─────────────────────────────────────────────────────────────
metadata = {
    "model_file":     CFG.OUTPUT_MODEL_PATH,
    "base_model":     CFG.BASE_MODEL,
    "num_classes":    CFG.NUM_CLASSES,
    "class_names":    list(CFG.PPE_CLASSES.values()),
    "imgsz":          CFG.IMGSZ,
    "epochs_trained": CFG.EPOCHS,
    "batch_size":     CFG.BATCH_SIZE,
    "optimizer":      CFG.OPTIMIZER,
    "lr0":            CFG.LR0,
    "cos_lr":         CFG.COS_LR,
    "val_mAP50":      round(float(val_metrics.box.map50),  4),
    "val_mAP50-95":   round(float(val_metrics.box.map),    4),
    "test_mAP50":     round(float(test_metrics.box.map50), 4),
    "test_mAP50-95":  round(float(test_metrics.box.map),   4),
    "train_images":   len(glob.glob(os.path.join(CFG.WORKING_PATH, "train", "images", "*"))),
    "valid_images":   len(glob.glob(os.path.join(CFG.WORKING_PATH, "valid", "images", "*"))),
    "test_images":    len(glob.glob(os.path.join(CFG.WORKING_PATH, "test",  "images", "*"))),
}

with open(CFG.METADATA_PATH, "w") as fp:
    json.dump(metadata, fp, indent=4)
print(f"📄 Metadata saved → {CFG.METADATA_PATH}")
print(json.dumps(metadata, indent=4))


## 🖥️ Interactive Inference Interface (Gradio)

Upload any construction-site image to get instant PPE compliance detection.  
- **Green boxes** → PPE correctly worn  
- **Red boxes** → PPE missing (safety violation)  
- The **compliance report** lists which items are worn and which are absent.

In [ ]:
import gradio as gr
from ultralytics import YOLO as _YOLO

# ── Load model once ───────────────────────────────────────────────────────────
_ppe_model = _YOLO(CFG.OUTPUT_MODEL_PATH)

CONF_THRESHOLD = 0.35
IOU_THRESHOLD  = 0.45


def predict_ppe(image: np.ndarray, conf: float = CONF_THRESHOLD) -> tuple[np.ndarray, str]:
    """
    Run PPE detection on a single image.

    Parameters
    ----------
    image : np.ndarray   RGB image from Gradio
    conf  : float        Detection confidence threshold

    Returns
    -------
    annotated_image : np.ndarray
    report          : str
    """
    if image is None:
        return None, "⚠️  No image provided."

    results  = _ppe_model(image, conf=conf, iou=IOU_THRESHOLD, verbose=False)
    annotated, compliance = draw_predictions(image, results[0])

    # ── Build compliance report ────────────────────────────────────────────────
    wearing_set = set(compliance["wearing"])
    missing_set = set(compliance["missing"])

    lines = ["## PPE Compliance Report\n"]

    if wearing_set:
        lines.append("### ✅ PPE Detected (worn correctly)")
        for item in sorted(wearing_set):
            lines.append(f"- {item}")
        lines.append("")

    if missing_set:
        lines.append("### ❌ Safety Violations (PPE missing)")
        for item in sorted(missing_set):
            lines.append(f"- {item}")
        lines.append("")

    overall = "🟢 **COMPLIANT** — All critical PPE present." if not missing_set \
              else "🔴 **NON-COMPLIANT** — Immediate action required."
    lines.append(f"\n**Overall Status:** {overall}")

    report = "\n".join(lines)
    return annotated, report


# ── Example images from test set ─────────────────────────────────────────────
example_images = random.sample(
    glob.glob(os.path.join(CFG.WORKING_PATH, "test", "images", "*")),
    min(4, len(glob.glob(os.path.join(CFG.WORKING_PATH, "test", "images", "*"))))
)

# ── Build Gradio UI ───────────────────────────────────────────────────────────
with gr.Blocks(title="PPE Kit Detection", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🦺 PPE Kit Detection — Construction Site Safety
        Detect Personal Protective Equipment (PPE) in construction site images using **YOLO11n**.
        Upload an image and click **Detect PPE** to see the analysis.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            input_image  = gr.Image(label="Input Image", type="numpy", height=480)
            conf_slider  = gr.Slider(0.1, 0.9, value=CONF_THRESHOLD, step=0.05,
                                     label="Confidence Threshold")
            detect_btn   = gr.Button("🔍 Detect PPE", variant="primary")

        with gr.Column(scale=1):
            output_image  = gr.Image(label="Annotated Output", height=480)
            output_report = gr.Markdown(label="Compliance Report")

    detect_btn.click(
        fn=predict_ppe,
        inputs=[input_image, conf_slider],
        outputs=[output_image, output_report],
    )

    if example_images:
        gr.Examples(
            examples=[[p, CONF_THRESHOLD] for p in example_images],
            inputs=[input_image, conf_slider],
            outputs=[output_image, output_report],
            fn=predict_ppe,
            cache_examples=False,
            label="Example Images from Test Set",
        )

    gr.Markdown(
        """
        ---
        **Classes detected:**  
        🟢 *Helmet, Gloves, Vest, Boots, Goggles, Person*  
        🔴 *no_helmet, no_goggle, no_gloves, no_boots*  
        """
    )

demo.launch(share=True, debug=False)


## 🏁 Conclusion

### What We Built
An end-to-end **YOLO11n** PPE detection system trained on the **Ketaki Construction Site** dataset.

### Pipeline Summary

| Step | Key Details |
|------|-------------|
| Dataset | Ketaki PPE — 11 classes (6 positive, 4 negative, 1 neutral) |
| Preprocessing | Stratified 75 / 15 / 10 re-split, pHash de-duplication |
| Augmentation | HFlip, Brightness/Contrast, HSV, GaussNoise for rare classes |
| Model | YOLO11n — `yolo11n.pt` base, AdamW, cosine LR, 50 epochs |
| Evaluation | Per-class P / R / mAP50 / mAP50-95 on val & test |
| Interface | Gradio app with live confidence slider + compliance report |

### Key Design Choices
- **Stratified split** ensures all 11 classes appear proportionally in every subset.
- **pHash deduplication** removes near-identical frames that inflate accuracy.
- **Colour-coded boxes** give an instant visual of compliance (green) vs violation (red).
- **Gradio `share=True`** generates a public Kaggle-accessible link on each run.

### References
1. Jocher, G. et al. *Ultralytics YOLO11* (2024). https://docs.ultralytics.com  
2. Ketaki PPE Dataset — Kaggle: `ppe-kit-detection-construction-site-workers`  
3. Buslaev, A. et al. *Albumentations: Fast and Flexible Image Augmentations*. Information 11(2), 125 (2020).  

---
*Notebook maintained as part of a Kaggle Computer Vision project portfolio.*